In [2]:
import numpy as np
import pandas as pd
import joblib
from tensorflow.keras.models import load_model

# Load everything
encoder = load_model("../data/kaggle-drdataboston/step_encoder_5step_no_overlap.keras")
scaler = joblib.load("../data/kaggle-drdataboston/step_scaler_5step_no_overlap.pkl")
df = pd.read_csv("../data/kaggle-drdataboston/all_5step_windows_no_overlap.csv")

# Drop metadata, keep only signal columns
X = df.iloc[:, 1:101].values  # Assuming column 0 is filename
X_scaled = scaler.transform(X)
X_encoded = encoder.predict(X_scaled)

print("✅ Encoded shape:", X_encoded.shape)  # e.g. (3420, 16)


2025-04-04 09:08:46.572834: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


564/564 ━━━━━━━━━━━━━━━━━━━━ 0s 594us/step
✅ Encoded shape: (18038, 16)


In [3]:
import pandas as pd
import numpy as np
import os

base_path = '../data/kaggle-drdataboston/'
windows = pd.read_csv(os.path.join(base_path, "all_5step_windows_no_overlap.csv"))
matrix = pd.read_csv(os.path.join(base_path, "matrix.csv"))

meta_columns = ['Weight', 'Age', 'Height (CM)', 'gender']
file_columns = [
    'subject_left_waist_session1',
    'subject_right_pocket_session1',
    'subject_left_waist_session2',
    'subject_right_pocket_session2'
]

metadata_rows = []

for fname in windows['filename']:
    matched_row = None
    for _, row in matrix.iterrows():
        if fname in row[file_columns].values:
            matched_row = row
            break

    if matched_row is not None:
        metadata_rows.append([fname] + [matched_row[col] for col in meta_columns])
    else:
        metadata_rows.append([fname] + [np.nan] * len(meta_columns))

# Save to metadata CSV
meta_df = pd.DataFrame(metadata_rows, columns=['filename'] + meta_columns)
meta_df.to_csv(os.path.join(base_path, "5step_metadata_no_overlap.csv"), index=False)
print("✅ Saved reconstructed metadata to 5step_metadata_no_overlap.csv")


✅ Saved reconstructed metadata to 5step_metadata_no_overlap.csv


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from tensorflow.keras.models import load_model
import joblib
import os

# Paths
base_path = '../data/kaggle-drdataboston/'
windows = pd.read_csv(os.path.join(base_path, "all_5step_windows_no_overlap.csv"))
metadata = pd.read_csv(os.path.join(base_path, "5step_metadata_no_overlap.csv"))
encoder = load_model(os.path.join(base_path, "step_encoder_5step_no_overlap.keras"))
scaler = joblib.load(os.path.join(base_path, "step_scaler_5step_no_overlap.pkl"))

# Filter out missing age
valid_idx = ~metadata['Age'].isna()
X = windows.loc[valid_idx, windows.columns[1:101]].values
meta = metadata.loc[valid_idx]

# Encode
X_scaled = scaler.transform(X)
X_encoded = encoder.predict(X_scaled)



564/564 ━━━━━━━━━━━━━━━━━━━━ 0s 538us/step


In [6]:
import numpy as np

np.save('../data/kaggle-drdataboston/X_encoded_5step_no_overlap.npy', X_encoded)
print("✅ Saved X_encoded to X_encoded.npy")


✅ Saved X_encoded to X_encoded.npy


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, classification_report

# Load encoded features + metadata
base_path = '../data/kaggle-drdataboston/'
X_encoded = np.load(os.path.join(base_path, 'X_encoded_5step_no_overlap.npy'))  # if you saved it as .npy
meta = pd.read_csv(os.path.join(base_path, '5step_metadata_5step_no_overlap.csv'))

# Make sure shape matches
assert len(meta) == len(X_encoded), "Mismatch between features and metadata"

# Filter out rows with missing values in any of the targets
targets = ['Age', 'Weight', 'Height (CM)', 'gender']
meta = meta.dropna(subset=targets)
X = X_encoded[meta.index]

# Encode gender as binary
meta['gender'] = meta['gender'].astype('category')
y_gender = meta['gender'].cat.codes
gender_labels = dict(enumerate(meta['gender'].cat.categories))

# Train-test split
X_train, X_test, y_gender_train, y_gender_test = train_test_split(X, y_gender, test_size=0.2, random_state=42)
_, _, y_age_train, y_age_test = train_test_split(X, meta['Age'], test_size=0.2, random_state=42)
_, _, y_weight_train, y_weight_test = train_test_split(X, meta['Weight'], test_size=0.2, random_state=42)
_, _, y_height_train, y_height_test = train_test_split(X, meta['Height (CM)'], test_size=0.2, random_state=42)

# Gender classification
clf_gender = RandomForestClassifier()
clf_gender.fit(X_train, y_gender_train)
y_pred_gender = clf_gender.predict(X_test)
print("🎯 Gender classification report:")
print(classification_report(y_gender_test, y_pred_gender, target_names=gender_labels.values()))

# Age prediction
reg_age = RandomForestRegressor()
reg_age.fit(X_train, y_age_train)
y_pred_age = reg_age.predict(X_test)
print(f"🎯 Age → R²: {r2_score(y_age_test, y_pred_age):.3f}, MAE: {mean_absolute_error(y_age_test, y_pred_age):.2f}")

# Weight prediction
reg_weight = RandomForestRegressor()
reg_weight.fit(X_train, y_weight_train)
y_pred_weight = reg_weight.predict(X_test)
print(f"🎯 Weight → R²: {r2_score(y_weight_test, y_pred_weight):.3f}, MAE: {mean_absolute_error(y_weight_test, y_pred_weight):.2f}")

# Height prediction
reg_height = RandomForestRegressor()
reg_height.fit(X_train, y_height_train)
y_pred_height = reg_height.predict(X_test)
print(f"🎯 Height → R²: {r2_score(y_height_test, y_pred_height):.3f}, MAE: {mean_absolute_error(y_height_test, y_pred_height):.2f}")


🎯 Gender classification report:
              precision    recall  f1-score   support

           F       0.54      0.58      0.56      8426
           M       0.60      0.57      0.58      9503

    accuracy                           0.57     17929
   macro avg       0.57      0.57      0.57     17929
weighted avg       0.58      0.57      0.57     17929

🎯 Age → R²: -0.014, MAE: 4.89
🎯 Weight → R²: 0.133, MAE: 12.01
🎯 Height → R²: 0.040, MAE: 7.59


In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models, callbacks
import os

# Load data
base_path = '../data/kaggle-drdataboston/'
X_encoded = np.load(os.path.join(base_path, 'X_encoded_5step_no_overlap.npy'))  # if you saved it as .npy
meta = pd.read_csv(os.path.join(base_path, '5step_metadata_no_overlap.csv'))

# Drop rows with any missing target
meta = meta.dropna(subset=['Age', 'Weight', 'Height (CM)', 'gender'])
X = X_encoded[meta.index]

# Encode gender as binary
meta['gender'] = meta['gender'].astype('category')
y_gender = meta['gender'].cat.codes
gender_labels = dict(enumerate(meta['gender'].cat.categories))

# Train-test split
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)
y_gender_train, y_gender_test = train_test_split(y_gender, test_size=0.2, random_state=42)
y_age_train, y_age_test = train_test_split(meta['Age'], test_size=0.2, random_state=42)
y_weight_train, y_weight_test = train_test_split(meta['Weight'], test_size=0.2, random_state=42)
y_height_train, y_height_test = train_test_split(meta['Height (CM)'], test_size=0.2, random_state=42)


In [18]:
def build_regression_model(input_dim):
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation='relu'),
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(1)  # regression output
    ])
    model.compile(optimizer='adam', loss='mae', metrics=['mae'])
    return model

def build_classification_model(input_dim):
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation='relu'),
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model


In [ ]:
es = callbacks.EarlyStopping(patience=20, restore_best_weights=True)

# Gender (Classification)
model_gender = build_classification_model(X.shape[1])
model_gender.fit(X_train, y_gender_train, validation_data=(X_test, y_gender_test),
                 epochs=60, batch_size=64, callbacks=[es])

# Age
model_age = build_regression_model(X.shape[1])
model_age.fit(X_train, y_age_train, validation_data=(X_test, y_age_test),
              epochs=60, batch_size=64, callbacks=[es])

# Weight
model_weight = build_regression_model(X.shape[1])
model_weight.fit(X_train, y_weight_train, validation_data=(X_test, y_weight_test),
                 epochs=60, batch_size=64, callbacks=[es])

# Height
model_height = build_regression_model(X.shape[1])
model_height.fit(X_train, y_height_train, validation_data=(X_test, y_height_test),
                 epochs=60, batch_size=64, callbacks=[es])


Epoch 1/60
451/451 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.5148 - loss: 0.6928 - val_accuracy: 0.5463 - val_loss: 0.6830
Epoch 2/60
451/451 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.5429 - loss: 0.6850 - val_accuracy: 0.5590 - val_loss: 0.6816
Epoch 3/60
451/451 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.5663 - loss: 0.6793 - val_accuracy: 0.5909 - val_loss: 0.6697
Epoch 4/60
451/451 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.5720 - loss: 0.6734 - val_accuracy: 0.5765 - val_loss: 0.6694
Epoch 5/60
451/451 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.5768 - loss: 0.6686 - val_accuracy: 0.5915 - val_loss: 0.6668
Epoch 6/60
451/451 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.5773 - loss: 0.6714 - val_accuracy: 0.5868 - val_loss: 0.6657
Epoch 7/60
451/451 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.5804 - loss: 0.6652 - val_accuracy: 0.5795 - val_loss: 0.6700
Epoch 8/60
451/451 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.5850 - loss: 0.6668 - val_accuracy: 0.

KeyboardInterrupt: 

In [17]:


# Predict & evaluate
from sklearn.metrics import r2_score, mean_absolute_error

y_age_pred = model_age.predict(X_test).flatten()
print(f"Age → R²: {r2_score(y_age_test, y_age_pred):.3f}, MAE: {mean_absolute_error(y_age_test, y_age_pred):.2f}")

y_weight_pred = model_weight.predict(X_test).flatten()
print(f"Weight → R²: {r2_score(y_weight_test, y_weight_pred):.3f}, MAE: {mean_absolute_error(y_weight_test, y_weight_pred):.2f}")

y_height_pred = model_height.predict(X_test).flatten()
print(f"Height → R²: {r2_score(y_height_test, y_height_pred):.3f}, MAE: {mean_absolute_error(y_height_test, y_height_pred):.2f}")

y_gender_pred = model_gender.predict(X_test).flatten() > 0.5
from sklearn.metrics import classification_report
print(classification_report(y_gender_test, y_gender_pred, target_names=gender_labels.values()))


113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 848us/step
Age → R²: -0.107, MAE: 4.40
113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 842us/step
Weight → R²: 0.098, MAE: 12.11
113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
Height → R²: 0.026, MAE: 7.43
113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 804us/step
              precision    recall  f1-score   support

           F       0.56      0.56      0.56      1672
           M       0.62      0.62      0.62      1936

    accuracy                           0.59      3608
   macro avg       0.59      0.59      0.59      3608
weighted avg       0.59      0.59      0.59      3608



In [ ]:

# Save all trained models
model_gender.save(os.path.join(base_path, "predict_gender.keras"))
model_age.save(os.path.join(base_path, "predict_age.keras"))
model_weight.save(os.path.join(base_path, "predict_weight.keras"))
model_height.save(os.path.join(base_path, "predict_height.keras"))

print("✅ All models saved in:", base_path)

gender
M    0.527878
F    0.472122
Name: proportion, dtype: float64
